Partially commutative monoids
trace monoids

There is a KB to do here that is in between multiset KB and string KB.

This is a slightly better fit for assembly I'd think. data constraints.


smart choices of fresh symbol generation. AC KB.
Compression


Telescope unification. Is kind of finding an overlap on each other

foata normal form

https://www2.informatik.uni-stuttgart.de/fmi/ti/veroeffentlichungen/pdffiles/DiekertMuscholl2011.pdf hmm. stack algorithm? I don't get it

https://doc.sagemath.org/html/en/reference/monoids/sage/monoids/trace_monoid.html

Do you only have capacity to do one a at a time? Multiset vs set storage

a commutes with itself or no?

Yea, so I have a different definition

In sorted version, is the level marked by a decreasing symbol?

overlap. partial overlap in head and tail layers (multiset), but inner layers have to match entirely? On paper first?

Exact string of sets match in the middle, but mutlsiet match on the ends.


Maybe I could implement this as type list[T] string knutg bendix where T implements overlap and equal?

What about   join == == == join

Nested string on multiset is kind of like the tower of theories idea.

Nested sequences of strings
[["a", "b"], ["c"]]  [["b"], ["c"] would have to know we're comin in from the left.
I guess that embeds into string rewriting with special $ delimiter
ab$c  b$c

sequence of polynomials would be about the same


In [ ]:
# https://www.philipzucker.com/string_knuth/
def overlaps(s,t):
    """critical pairs https://en.wikipedia.org/wiki/Critical_pair_(term_rewriting)"""
    # make len(t) >= len(s)
    if len(t) < len(s):
        s,t = t,s
    if subseq(s,t) is not None:
        yield t
    # iterate over possible overlap sizes 1 to the len(s) at edges
    for osize in range(1,len(s)):
        if t[-osize:] == s[:osize]:
            yield t + s[osize:]
        if s[-osize:] == t[:osize]:
            yield s + t[osize:]

In [13]:
from collections import Counter

def subms(xs, ys):
    cx, cy = Counter(xs), Counter(ys)
    return all(cx[a] <= cy[a] for a in cx)

assert subms([1,2,3], [1,2,3,4])
assert not subms([1,2,3,4], [1,2,3])

In [ ]:
type Trace = list[list[str]]


def subtrace(s : Trace, t : Trace) -> list[Trace]:
    res = []
    if len(s) == 1:
        for i in range(len(t)):
            if subms(s[0], t[i]):
                res.append(i)
    else:
        for i in range(len(t) - len(s) + 1):
            if subms(s[0], t[i]) and s[1:-1] == t[i+1:i+len(s)-1] and subms(s[-1], t[i+len(s)-1]):
                res.append(i)
    return res

    
assert subtrace([["a", "b"], ["c"]], [["a", "b"], ["c"], ["d"]]) == [0]
assert subtrace([["a", "b"], ["c"]], [["a", "b"], ["d"], ["c"]]) == []
assert subtrace([["a", "b"], ["c"]], [["a", "b"], ["c"]]) == [0]

In [ ]:
def overlaps(s, t):
    for osize in range(1,len(s)):
        t[:osize]

In [ ]:
def subms(s, t):

def subseq(s,t):
    """Return index when s is a subsequence of t, None otherwise"""
    for i in range(len(t) - len(s) + 1):
        if s == t[i:i+len(s)]:
            return i
    return None

In [20]:
def ms_union(xs, ys):
    """Pointwise max union of multisets."""
    return Counter(xs) | Counter(ys)
ms_union([1,2,3], [2,3,4])

Counter({1: 1, 2: 1, 3: 1, 4: 1})

In [ ]:
def pmatch(s, p):
    

In [ ]:
def non_triv_overlap(s, i : int, t):
    #if i == 0:
    #    not (i == 0) and non_triv_ms(s[0], t[0]) and s[1:min(len(s), len(t))] == t[1:min(len(s), len(t))] and non_triv_ms(s[-1], t[-1])
    not (i == 0) or non_triv_ms(s[0], t[0])
    not (i != 0) or subms(s[i], t[0])
    not (i + len(t) == len(s)) or non_triv_ms(s[-1], t[-1]) # t ends at end of s
    not (i + len(t) > len(s)) or subms(s[-1], t[len(s) - i - 1]) # t is extends beyond
    not (i + len(t) < len(s)) or subms(t[-1], s[i + len(t)]) # t is contained in s
    t[1:min(len(t)-1, len(s) - i)] == s[i+1:i+min(len(t)-1, len(s) - i)] # middle of t is contained in s


In [ ]:
def merge_at(s, i : int, t):
    assert len(t) != 0
    if len(t) == 1:
        if i == 0 or i == len(s) - 1:
            res = s.copy()
            res[i] = ms_union(s[i], t[0])
            return res
    else:
        
        # but also if they end on the same end is special, we can use the multiset overlap then
        
        # deal with up to i
        if i == 0:
            res = [ms_union(s[0], t[0])]
        else:
            res = s[:i]
            if not subms(t[0], s[i]):
                return None
            res.append(s[i])
        for j in range(j, len(t) - 1):
            if i + j < len(s):
                if s[i+j] != t[j]:
                    return None
            elif i + j == len(s):
                if not subms(s[i+j], t[j]):
                    return None
            else:
                pass # we are beyond s now
            res.append(t[j])
        #res.extend(t[1:-1]) # add in all of t except the first and last
        n = i + len(t) - 1
        if n == len(s):
            res.append(ms_union(s[-1], t[-1]))
            return res
        else:
            if not subms(t[-1], s[n]):
                return None
            res.append(s[n])
        if len(s) > n:
            res.extend(s[n+1:])
        return res

            
        if i + len(t) > len(s): # goes off the end
            if not subms(t[0], s[i]) or not subms(s[-1], t[len(s) - i]) or not all(t[i] == s[i] for i in range(len(s) - i - 1)):
                return None
            res = s.copy()
            res.pop()
            res.extend(t[len(s) - i:])
            return res
        else:
            # t contained in s
            if not subms(t[0], s[i]) or not subms(t[-1], s[i + len(t) - 1]):
                return None
            return s.copy()

def overlaps(s,t):
    res = []
    for i in range(len(s)):
        m = merge_at(s, i, t)
        if m is not None:
            res.append(m)
    for j in range(len(t)):
        m = merge_at(t, j, s)
        if m is not None:
            res.append(m)
    return res



In [2]:
commutes = {("a","b"), ("b", "a")}
def foata_slow(s):
    levels = []
    for i,c in enumerate(s):
        level = 0
        for j,y in enumerate(s[:i]):
            if y != c and (y,c) not in commutes:
                level = max(level, levels[j] + 1)
        levels.append(level)
    return levels

foata_slow("cabad")




[0, 1, 1, 1, 2]

depends and commutes are kind of opposite data.

In [10]:
# Things we can't commute through
depends = {
    "a": {"c"}, 
    "c": {"a", "b"},
    "b": {"c"},
    "d":{"c", "a", "b"}
}

def foata(s):
    last_levels = {c : -1 for c in "abcd"}
    levels = []
    for i,c in enumerate(s):
        level = max((last_levels[y] for y in depends[c]), default=-1) + 1
        levels.append(level)
        last_levels[c] = level
    return levels

foata("cabad")


def layer(s, levels):
    layers = [[] for _ in range(max(levels)+1)]
    for c, l in zip(s, levels):
        layers[l].append(c)
    return [sorted(layer) for layer in layers]
layer("cabad", foata("cabad"))


[['c'], ['a', 'a', 'b'], ['d']]

In [ ]:
def bubbin(s):
    for i in range(len(s)):
        for j in range(len(s)-1):
            if (s[j], s[j+1]) in commutes:
                s = s[:j] + s[j+1] + s[j] + s[j+2:]